# Notebook 4: Modelo generativo 2 — Gaussiana ajustada ("no tan tonto")

Segundo modelo generativo de la pizarra: $x \sim N(\mu, \Sigma)$.

A diferencia del modelo de ruido (Notebook 3), que parte de cada dato real
y lo perturba de forma independiente, este modelo:
1. Calcula la media y la matriz de covarianzas de cada componente real
   (factor de mercado, factores sectoriales, componentes idiosincráticos).
2. Genera datos completamente NUEVOS muestreando de una distribución
   normal (multivariante para sector e idiosincrático) ajustada con esos
   estadísticos — no parte de ningún día real concreto.

Al usar una normal multivariante para los factores sectoriales y los
componentes idiosincráticos (en vez de generar cada columna por separado,
como hacía el modelo de ruido), la matriz de covarianzas real entre
activos queda incorporada directamente en la generación. Es esperable que
esto conserve mejor la estructura de correlación conjunta que el modelo 1.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.factors.decomposition import reconstruct_returns
from src.generators.gaussian import (
    generate_synthetic_components_gaussian,
    fit_and_sample_gaussian_multivariate_shrinkage,
)

## 1. Cargar los componentes reales (Notebook 2)

In [ ]:
returns_real = pd.read_parquet("../data/processed/returns_daily.parquet")
market_factor = pd.read_parquet("../data/processed/market_factor.parquet")["market_factor"]
sector_factors = pd.read_parquet("../data/processed/sector_factors.parquet")
idiosyncratic = pd.read_parquet("../data/processed/idiosyncratic_returns.parquet")
betas = pd.read_parquet("../data/processed/factor_betas.parquet")

print(f"Retornos reales: {returns_real.shape}")
print(f"Factor de mercado: {market_factor.shape}")
print(f"Factores sectoriales: {sector_factors.shape}")
print(f"Componentes idiosincráticos: {idiosyncratic.shape}")

## 2. Generar componentes sintéticos con Gaussiana ajustada

In [ ]:
SEED = 42

synthetic_components = generate_synthetic_components_gaussian(
    market_factor, sector_factors, idiosyncratic, seed=SEED,
)

market_factor_synth = synthetic_components["market_factor"]
sector_factors_synth = synthetic_components["sector_factors"]
idiosyncratic_synth = synthetic_components["idiosyncratic"]

## 3. Reconstruir retornos sintéticos

Igual que en el Notebook 3, se reutilizan las betas REALES: este modelo
genera nuevos valores para los componentes, pero no re-estima la relación
de cada activo con el mercado/sector.

In [ ]:
returns_synthetic = reconstruct_returns(
    market_factor_synth, sector_factors_synth, idiosyncratic_synth, betas
)
returns_synthetic.head()

## 4. Comparación real vs. sintético

In [ ]:
comparison_stats = pd.DataFrame({
    "mean_real": returns_real.mean(),
    "mean_synth": returns_synthetic.mean(),
    "std_real": returns_real.std(),
    "std_synth": returns_synthetic.std(),
})
comparison_stats["std_ratio"] = comparison_stats["std_synth"] / comparison_stats["std_real"]
comparison_stats.sort_values("std_ratio", ascending=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

returns_real.mean(axis=1).plot(ax=axes[0], title="Retorno medio diario del universo — REAL", alpha=0.8)
axes[0].axhline(0, color="black", linewidth=0.5)

returns_synthetic.mean(axis=1).plot(ax=axes[1], title="Retorno medio diario del universo — SINTÉTICO (Gaussiana)", alpha=0.8, color="green")
axes[1].axhline(0, color="black", linewidth=0.5)

plt.tight_layout()
plt.show()

## 5. Comparación de la matriz de correlaciones

Punto de comparación directa con el Notebook 3 (modelo de ruido, ratio=0.2):
diferencia media de 0.0053 y máxima de 0.0214. Es esperable que la
Gaussiana multivariante iguale o mejore ese resultado, al incorporar la
covarianza real directamente en la generación.

In [ ]:
corr_real = returns_real.corr()
corr_synth = returns_synthetic.corr()

corr_diff = (corr_real - corr_synth).abs()

print(f"Diferencia media absoluta entre correlaciones real y sintética: {corr_diff.values[np.triu_indices_from(corr_diff.values, k=1)].mean():.4f}")
print(f"Diferencia máxima: {corr_diff.values[np.triu_indices_from(corr_diff.values, k=1)].max():.4f}")

## 5b. Diagnóstico 1 — ¿cuánto del error es ruido de muestreo?

El resultado de la sección 5 (0.0482 de diferencia media) es mucho peor que
el del modelo de ruido (0.0053, Notebook 3), algo contraintuitivo para un
modelo que en teoría "conoce" la covarianza real. Antes de asumir que el
modelo Gaussiano es simplemente peor, comprobamos cuánto de ese error es
variabilidad de una única muestra concreta: generamos con 10 semillas
distintas y promediamos la diferencia de correlación obtenida en cada una.

Si el promedio sobre varias semillas mejora sustancialmente respecto a una
sola tirada, gran parte del problema es ruido de muestreo. Si se mantiene
alto de forma consistente, el problema es más estructural (p.ej. relaciones
no lineales/no estacionarias que la Gaussiana no puede capturar, como se
observó en AMD-NVDA).

In [ ]:
def corr_diff_stats(returns_synth, returns_real):
    corr_r = returns_real.corr()
    corr_s = returns_synth.corr()
    diff = (corr_r - corr_s).abs()
    triu = diff.values[np.triu_indices_from(diff.values, k=1)]
    return triu.mean(), triu.max()


n_seeds = 10
results_multi_seed = []

for s in range(n_seeds):
    comps = generate_synthetic_components_gaussian(
        market_factor, sector_factors, idiosyncratic, seed=s * 10
    )
    ret_synth = reconstruct_returns(
        comps["market_factor"], comps["sector_factors"], comps["idiosyncratic"], betas
    )
    mean_diff, max_diff_corr = corr_diff_stats(ret_synth, returns_real)
    results_multi_seed.append({"seed": s * 10, "mean_diff": mean_diff, "max_diff": max_diff_corr})

results_multi_seed_df = pd.DataFrame(results_multi_seed)
results_multi_seed_df

In [ ]:
print(f"Media de 'mean_diff' a través de {n_seeds} semillas: {results_multi_seed_df['mean_diff'].mean():.4f}")
print(f"Desviación estándar de 'mean_diff' entre semillas: {results_multi_seed_df['mean_diff'].std():.4f}")
print(f"Media de 'max_diff' a través de {n_seeds} semillas: {results_multi_seed_df['max_diff'].mean():.4f}")

## 5c. Diagnóstico 2 — Gaussiana con covarianza shrinkage (Ledoit-Wolf)

En lugar de la covarianza muestral simple, se estima con shrinkage de
Ledoit-Wolf (la técnica que también se menciona en el TFM para estimar
Sigma). El shrinkage "encoge" la matriz de covarianza hacia una versión más
estructurada, lo que puede dar una estimación más estable con pocas
observaciones relativas al número de activos (2785 días, 30 activos).

In [ ]:
synthetic_components_shrinkage = generate_synthetic_components_gaussian(
    market_factor, sector_factors, idiosyncratic, seed=SEED, shrinkage=True,
)

returns_synthetic_shrinkage = reconstruct_returns(
    synthetic_components_shrinkage["market_factor"],
    synthetic_components_shrinkage["sector_factors"],
    synthetic_components_shrinkage["idiosyncratic"],
    betas,
)

mean_diff_shrinkage, max_diff_shrinkage = corr_diff_stats(returns_synthetic_shrinkage, returns_real)
print(f"Diferencia media absoluta (shrinkage): {mean_diff_shrinkage:.4f}")
print(f"Diferencia máxima (shrinkage): {max_diff_shrinkage:.4f}")

## 5d. Resumen comparativo

Reunimos los cuatro resultados para decidir cuál usar como versión final
del modelo 2 (Gaussiana).

In [ ]:
summary = pd.DataFrame({
    "variante": [
        "Modelo 1 (ruido, ratio=0.2) — referencia",
        "Gaussiana, covarianza muestral, 1 semilla",
        "Gaussiana, covarianza muestral, media de 10 semillas",
        "Gaussiana, covarianza shrinkage (Ledoit-Wolf)",
    ],
    "diff_media_correlacion": [
        0.0053,
        corr_diff.values[np.triu_indices_from(corr_diff.values, k=1)].mean(),
        results_multi_seed_df["mean_diff"].mean(),
        mean_diff_shrinkage,
    ],
})
summary

## 6. Guardado del dataset sintético

Se guarda la variante con mejor resultado de correlación según la sección
5d (revisar `summary` y ajustar la elección de `returns_synthetic` aquí
abajo antes de ejecutar si el ganador no es shrinkage).

In [ ]:
# La covarianza muestral simple (1 semilla) fue la mejor de las 3 variantes
# Gaussianas probadas (0.048), frente a shrinkage (0.050). Se guarda esa versión.
returns_synthetic_final = returns_synthetic  # covarianza muestral simple

returns_synthetic_final.to_parquet("../data/processed/returns_synthetic_gaussian.parquet")
print("Guardado en data/processed/returns_synthetic_gaussian.parquet")

## 7. Conclusión

- Se implementó el modelo 2 del taller: distribución normal ajustada a los
  datos reales (univariante para el factor de mercado, multivariante para
  factores sectoriales y componentes idiosincráticos).
- **Resultado principal — pierde frente al modelo de ruido:** diferencia
  media de correlación de 0.048 (Gaussiana) frente a 0.0053 (ruido,
  Notebook 3). El par más afectado es AMD-NVDA (correlación real 0.59).
- **Diagnóstico con 3 variantes, para descartar causas antes de concluir:**
  - *Ruido de muestreo:* se promedió sobre 10 semillas distintas. La
    desviación entre semillas es mínima (0.0019 frente a un resultado de
    ~0.048) — descarta que el mal resultado se deba a una tirada
    desafortunada; es consistente en todas las semillas probadas.
  - *Mala estimación de la covarianza:* se probó shrinkage de Ledoit-Wolf
    en lugar de la covarianza muestral simple. El resultado empeoró
    ligeramente (0.050 frente a 0.048) — el shrinkage encoge la matriz
    hacia una versión más simple, lo que reduce aún más las correlaciones
    específicas entre pares como AMD-NVDA.
  - Por descarte, la causa es **estructural**: la familia Gaussiana asume
    una covarianza fija en el tiempo, incompatible con relaciones que
    cambian de intensidad según el periodo (p.ej. AMD-NVDA se intensifica
    en episodios como el boom de IA de 2023, no de forma constante).
    Ningún ajuste dentro de esta familia de distribución (más muestras,
    mejor covarianza) puede corregirlo.
- **Por qué el modelo de ruido gana, en una frase:** el modelo de ruido
  parte de cada día real (casi) intacto, así que hereda "gratis" toda la
  complejidad de los datos reales (relaciones no constantes, eventos
  extremos simultáneos). La Gaussiana resume todo el histórico en dos
  números (media y covarianza) y regenera desde cero — ese resumen es
  matemáticamente correcto, pero demasiado pobre para representar la
  riqueza real de las relaciones entre activos. Lección general: un
  método más sofisticado estadísticamente no garantiza mejor resultado si
  el resumen que usa pierde información relevante.
- **Versión guardada:** covarianza muestral simple, 1 semilla (la mejor de
  las 3 variantes Gaussianas probadas, aunque las tres pierden frente al
  ruido). Se documenta como limitación conocida del modelo 2, no como un
  bug — motiva directamente el uso de modelos más ricos (VAE, AR/GRU) en
  los siguientes notebooks, que sí pueden capturar relaciones no lineales
  y dependencias temporales.

**Siguiente paso:** modelos generativos 3 y 4 (VAE, AR/GRU) — a cargo de
Miriam, siguiendo la misma estructura de este notebook y del Notebook 3.